In [34]:
import pandas as pd
import seaborn as sns
import ast 

In [35]:

templates = pd.read_excel('../data/templates.xlsx', sheet_name='Templates')
prompts = pd.read_csv('../data/prompts.csv')

# Extraer el prefijo de cada template (texto antes del primer '[')
templates['prefix'] = templates['prompt'].apply(lambda t: t.split('[')[0])

# Asignar template_id y category a cada prompt según el prefijo
def match_template(prompt):
    for _, row in templates.iterrows():
        if prompt.startswith(row['prefix']):
            return pd.Series({'template_id': row['ID'], 'category': row['category']})
    return pd.Series({'template_id': None, 'category': None})

prompts[['template_id', 'category']] = prompts['prompt'].apply(match_template)

In [36]:
prompts

,prompt,template_id,category
0,"I want to write an article about: ""Common fair...",1,3
1,"I want to write an article about: ""Machine Lea...",1,3
2,"I want to write an article about: ""Evaluation ...",1,3
3,"I want to write an article about: ""Benchmark c...",1,3
4,"I want to write an article about: ""Word embedd...",1,3
...,...,...,...
2130,"Define ""Positive bias"". Include references.",11,4
2131,"Define ""Governance in AI"". Include references.",11,4
2132,"Define ""Intrinsic hallucinations"". Include ref...",11,4
2133,"Define ""Extrinsic hallucinations"". Include ref...",11,4


# Gemini

### URL existe

In [37]:
gemini_url_check = pd.read_csv('../data/analisis/processed/gemini_url_checks_progress.csv')
gemini_url_check['url_check'] = gemini_url_check['url_check'].apply(lambda x: ast.literal_eval(x))
gemini_url_check = gemini_url_check.explode("url_check")
gemini_url_check = gemini_url_check[['prompt', 'url_check']]
gemini_url_check['url_exists'] = gemini_url_check['url_check'].apply(lambda x: x['exists'])
                                                                     
gemini_url_check['url_exists'].value_counts()

url_exists
True     14755
False     1374
Name: count, dtype: int64

In [38]:
gemini_url_check = gemini_url_check.merge(prompts, on = 'prompt')
gemini_url_check.groupby('category')['url_exists'].value_counts()

category  url_exists
1         True          2953
          False          556
2         True          3705
          False          138
3         True          3924
          False          237
4         True          4173
          False          443
Name: count, dtype: int64

In [39]:
gemini_url_check.groupby('category')['url_exists'].value_counts(True)

category  url_exists
1         True          0.841550
          False         0.158450
2         True          0.964091
          False         0.035909
3         True          0.943043
          False         0.056957
4         True          0.904029
          False         0.095971
Name: proportion, dtype: float64

### Peer reviewed

In [40]:
gemini_results = pd.read_csv('../data/analisis/processed/gemini_url_classify.csv')
gemini_results.drop(columns = [c for c in gemini_results.columns if 'Unnamed' in c], inplace=True)
gemini_results['url'] = gemini_results.url_check.apply(lambda x: ast.literal_eval(x)['url'])
gemini_results['peer_reviewed'] = gemini_results.clasificacion.apply(lambda x: ast.literal_eval(x)['peer_reviewed'])

In [41]:
gemini_results = gemini_results[['prompt', 'url', 'peer_reviewed']]
gemini_results = gemini_results.merge(prompts, on = 'prompt')
gemini_results.peer_reviewed.value_counts()

peer_reviewed
no    12787
sí     1968
Name: count, dtype: int64

In [42]:
gemini_results.groupby('category')['peer_reviewed'].value_counts()

category  peer_reviewed
1         no               2557
          sí                396
2         no               3198
          sí                507
3         no               3364
          sí                560
4         no               3668
          sí                505
Name: count, dtype: int64

In [43]:
gemini_results.groupby('category')['peer_reviewed'].value_counts(True)

category  peer_reviewed
1         no               0.865899
          sí               0.134101
2         no               0.863158
          sí               0.136842
3         no               0.857288
          sí               0.142712
4         no               0.878984
          sí               0.121016
Name: proportion, dtype: float64

# Mistral

### APA

In [44]:
mistral_apa = pd.read_csv('../data/analisis/processed/mistral_reference_checks.csv')
mistral_apa = mistral_apa[['prompt', 'status']]
mistral_apa = mistral_apa.merge(prompts, on = 'prompt')
mistral_apa['status'].value_counts()

status
review          5586
hallucinated    3103
exists          2810
Name: count, dtype: int64

In [45]:
mistral_apa.groupby('category')['status'].value_counts()

category  status      
1         review          1350
          hallucinated     896
          exists           586
2         review          1108
          hallucinated     805
          exists           550
3         review          1832
          exists           817
          hallucinated     455
4         review          1296
          hallucinated     947
          exists           857
Name: count, dtype: int64

In [46]:
mistral_apa.groupby('category')['status'].value_counts(True)

category  status      
1         review          0.476695
          hallucinated    0.316384
          exists          0.206921
2         review          0.449858
          hallucinated    0.326837
          exists          0.223305
3         review          0.590206
          exists          0.263209
          hallucinated    0.146585
4         review          0.418065
          hallucinated    0.305484
          exists          0.276452
Name: proportion, dtype: float64

### URL

In [47]:
mistral_url = pd.read_csv('../data/analisis/processed/mistral_classify_complete.csv')
mistral_url['url'] = mistral_url.url_check.apply(lambda x: ast.literal_eval(x)['url'])
mistral_url['existe'] = mistral_url.url_check.apply(lambda x: ast.literal_eval(x)['exists'])
mistral_url = mistral_url[['prompt', 'url', 'existe']]
mistral_url = mistral_url.merge(prompts, on = 'prompt')

In [49]:
mistral_url.groupby('category')['existe'].value_counts(True)

category  existe
1         False     0.617647
          True      0.382353
2         False     0.629921
          True      0.370079
3         False     0.554545
          True      0.445455
4         False     0.682353
          True      0.317647
Name: proportion, dtype: float64

In [54]:
mistral_url_pr = pd.read_csv('../data/analisis/processed/mistral_classify.csv')
mistral_url_pr['url'] = mistral_url_pr.url_check.apply(lambda x: ast.literal_eval(x)['url'])
mistral_url_pr['peer_reviewed'] = mistral_url_pr.clasificacion.apply(lambda x: ast.literal_eval(x)['peer_reviewed'])
mistral_url_pr = mistral_url_pr[['prompt', 'url', 'peer_reviewed']]
mistral_url_pr = mistral_url_pr.merge(prompts, on = 'prompt')

In [55]:
mistral_url_pr.groupby('category')['peer_reviewed'].value_counts(True)

category  peer_reviewed
1         no               0.884615
          sí               0.115385
2         no               0.854167
          sí               0.145833
3         no               0.510204
          sí               0.489796
4         no               0.875000
          sí               0.125000
Name: proportion, dtype: float64

# GPT

### URL existe

In [16]:
gpt_url_check = pd.read_csv('../data/analisis/processed/results_gpt_full.csv')
gpt_url_check = gpt_url_check[['prompt', 'status_check']]
gpt_url_check['status_check'] = gpt_url_check.status_check.apply(lambda x: ast.literal_eval(x))
gpt_url_check = gpt_url_check.explode('status_check')

In [17]:
gpt_url_check.head()

,prompt,status_check
0,"I want to write an article about: ""Common fair...",{'url': 'https://jmlr.org/beta/papers/v24/22-1...
0,"I want to write an article about: ""Common fair...",{'url': 'https://arxiv.org/abs/1902.04783?utm_...
0,"I want to write an article about: ""Common fair...",{'url': 'https://arxiv.org/abs/1711.05144?utm_...
0,"I want to write an article about: ""Common fair...",{'url': 'https://arxiv.org/abs/2107.04642?utm_...
0,"I want to write an article about: ""Common fair...",{'url': 'https://www.emergentmind.com/papers/1...


In [18]:
gpt_url_check = gpt_url_check[~gpt_url_check.status_check.isna()]

In [19]:
gpt_url_check['exists'] = gpt_url_check['status_check'].apply(lambda x: x['exists'])
gpt_url_check = gpt_url_check.merge(prompts, on = 'prompt')
gpt_url_check['exists'].value_counts()

exists
True     7411
False     174
Name: count, dtype: int64